
# 02 – Train Model & View in TensorBoard

This notebook:
- Configures a run (data glob, split, model hyperparams)
- Trains `IrrMLPClassifier` via `run_train(cfg)`
- Shows last-epoch metrics from the CSV logger
- Locates the **best checkpoint** and prints its path
- Includes a one-liner to launch **TensorBoard**

> **Tip:** Edit the config cell to match your data and preferences.


In [ ]:

# ==== Config ====
DATA_GLOB   = "raw_data/*.csv"  # e.g., "raw_data/*training*.csv"
BATCH_SIZE  = 512
VAL_RATIO   = 0.20
SEED        = 88
GROUP_COL   = "h3_r7"           # e.g., "h3_r7", "county_fips", ".geo", or "none" for stratified

# Early stopping / training
MONITOR     = "val_auprc"
PATIENCE    = 10
MIN_DELTA   = 1e-5
MAX_EPOCHS  = 60

# Model hyperparameters
HIDDEN       = 256
DEPTH        = 2
DROPOUT      = 0.10
ACT          = "silu"           # one of: relu, silu, gelu
LR           = 1e-3
WEIGHT_DECAY = 1e-4
STANDARDIZE  = False            # embeddings are unit-length → usually False

# Where logs go (your code sets this internally, usually under outputs/logs/)
LOG_ROOT = "outputs/logs"


In [ ]:

# Imports
from pathlib import Path
import glob
import pandas as pd
import numpy as np
import torch

# Project imports
from irr.training.train import run_train
# Configs and model config (handle old/new names gracefully)
try:
    from irr.configs import TrainConfig as _TrainConfig
except Exception:
    from irr.configs import RunConfig as _TrainConfig  # if you kept an older alias
try:
    from irr.models.mlp_classifier import ModelConfig as _ModelConfig
except Exception:
    from irr.models.mlp_classifier import TinyCfg as _ModelConfig  # legacy alias

# Utility: find newest best.ckpt
def find_latest_best_ckpt(root="outputs/logs"):
    pattern = str(Path(root) / "**" / "checkpoints" / "best.ckpt")
    paths = glob.glob(pattern, recursive=True)
    if not paths:
        return None
    paths.sort(key=lambda p: Path(p).stat().st_mtime, reverse=True)
    return paths[0]

# Utility: read last-epoch metrics from a CSV logger directory
def read_last_epoch_metrics(csv_log_dir: str) -> pd.Series | None:
    metrics_csv = Path(csv_log_dir) / "metrics.csv"
    if not metrics_csv.exists():
        return None
    m = pd.read_csv(metrics_csv)
    if "epoch" not in m.columns and "step" in m.columns:
        m["epoch"] = m["step"]
    m = m.sort_values(["epoch"]).groupby("epoch").last()
    return m.iloc[-1] if len(m) else None


In [ ]:

# Build model config and training config
group_col = None if str(GROUP_COL).lower() == "none" else GROUP_COL

model_cfg = _ModelConfig(
    hidden=HIDDEN,
    depth=DEPTH,
    dropout=DROPOUT,
    act=ACT,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    standardize=STANDARDIZE,
)

cfg = _TrainConfig(
    data_glob=DATA_GLOB,
    batch_size=BATCH_SIZE,
    val_ratio=VAL_RATIO,
    seed=SEED,
    monitor=MONITOR,
    patience=PATIENCE,
    max_epochs=MAX_EPOCHS,
    # Some codebases also accept min_delta/group_col; if your dataclass includes them, set them here:
    # min_delta=MIN_DELTA,
    # group_col=group_col,
    model=model_cfg,
    group_col=group_col
)

print("Training with config:")
print(cfg)

result = run_train(cfg)
csv_log_dir = Path(result.get("log_dir", "outputs/logs"))
print(f"CSV logger directory: {csv_log_dir}")


In [ ]:

last = read_last_epoch_metrics(str(csv_log_dir))
if last is None:
    print("metrics.csv not found yet in CSV logger directory.")
else:
    display(last.to_frame().T)
    keys = ["val_auprc", "val_auroc", "train_loss", "val_loss"]
    print({k: float(last.get(k)) for k in keys if k in last.index})


In [ ]:

ckpt = find_latest_best_ckpt(LOG_ROOT)
if ckpt is None:
    print("Could not find best.ckpt under", LOG_ROOT)
else:
    print("Best checkpoint:", ckpt)



## View in TensorBoard

From your project root, run:

```bash
tensorboard --logdir outputs/logs --port 6006
```

Then open http://localhost:6006/ in your browser.
